# Game Understanding

Runs the two tasks of the first part of the project with the four models:

- **Explain**: explanation of the original rules to someone who has never played the game.
- **Flaw Detection**: finding missing, ambiguous or inconsistent rules in rulebooks with three injected flaws, and suggesting corrections.

In [ ]:
!pip install -q bitsandbytes accelerate

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
basePath = "/content/drive/My Drive/Progetto NLP 2026"
for folder in ["src", "data"]:
    path = os.path.join(basePath, folder)
    if path not in sys.path:
        sys.path.insert(0, path)
outPath = os.path.join(basePath, "results", "Results_game_understanding")
os.makedirs(outPath, exist_ok=True)

Mounted at /content/drive


In [ ]:
import json, gc
import torch
from tqdm.auto import tqdm

from models import Qwen, Llama1B, Qwen7B, Mistral7B
from dataset import games
from dataset_flaws import buildFlawedGames
from runTest import testGameUnderstanding, testGameFlawDetection

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("using device:", device)

flawedGames = buildFlawedGames(games)

using device: cuda


## Configuration

In [ ]:
models = [
    ("Qwen", Qwen),
    ("Llama-1B", Llama1B),
    ("Qwen-7B", Qwen7B),
    ("Mistral-7B", Mistral7B),
]
tasks = ["explain", "flaw_detection"]
runs = 3
baseSeed = 42
selectedGames = [
    "connect-four", "checkers", "nine-mens-morris",
    "ticket-to-ride", "kingdomino", "azul",
    "uno", "exploding-kittens", "mao",
    "monopoly", "clue", "forbidden-island",
    "werewolf", "dixit", "the_resistance",
]

## Generation

In [ ]:
def resultsPath(modelName):
    return os.path.join(outPath, f"results_{modelName.replace('-', '_')}.json")

# results are rewritten after every answer, through a temporary file,
# so that an interrupted session loses nothing
def save(path, rows):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)

for modelName, modelClass in models:
    path = resultsPath(modelName)

    # answers already saved are skipped
    rows = json.load(open(path, encoding="utf-8")) if os.path.exists(path) else []
    done = {(r["task"], r["game"], r["iteration"]) for r in rows}
    todo = [(task, game, run) for task in tasks for game in selectedGames for run in range(1, runs + 1)
            if (task, game, run) not in done]
    if not todo:
        print(modelName, ": done"); continue

    model = modelClass(device)
    print(f"{modelName}: max context {model.max_context} token, {len(todo)} to do")

    for task, gameName, run in tqdm(todo, desc=modelName):
        seed = baseSeed + run # 43, 44, 45: one seed per iteration
        if task == "explain":
            output = testGameUnderstanding(model, gameName, games[gameName], seed=seed)
        else:
            output = testGameFlawDetection(model, gameName, flawedGames[gameName], seed=seed)
        rows.append({"model": modelName, "iteration": run, **output})
        save(path, rows)
        torch.cuda.empty_cache()

    # free the GPU memory before loading the next model
    del model; gc.collect(); torch.cuda.empty_cache()

Qwen : done
Llama-1B : done
Qwen-7B : done
Mistral-7B : done


## Quick check
How many answers stopped at the `max_new_tokens` limit (answer cut off).

In [ ]:
import pandas as pd

rows = []
for fileName in sorted(os.listdir(outPath)):
    if fileName.endswith(".json"):
        rows.extend(json.load(open(os.path.join(outPath, fileName), encoding="utf-8")))
df = pd.DataFrame(rows)

print(df.groupby(["model", "task"]).size().unstack())
print("\nmax_new_tokens limit exceeded:")
print(df.groupby(["model", "task"])["hit_max_new_tokens"].mean().round(2).unstack())

task        explain  flaw_detection
model                              
Llama-1B         45              45
Mistral-7B       45              45
Qwen             45              45
Qwen-7B          45              45

max_new_tokens limit exceeded:
task        explain  flaw_detection
model                              
Llama-1B       0.00            0.02
Mistral-7B     0.00            0.02
Qwen           0.02            0.04
Qwen-7B        0.00            0.02
